# VP-SDE Probability-Flow ODE 与 DPM-Solver：从 SDE 到二阶多步公式

本笔记完整推导本项目 VP-SDE 实验的数学对象，而不是把 DPM-Solver 的公式当作黑盒。路线是：

$$
\text{VP forward SDE}
\;\Longrightarrow\;
\text{Fokker--Planck 方程}
\;\Longrightarrow\;
\text{probability-flow ODE}
\;\Longrightarrow\;
\lambda\text{ 空间半线性积分式}
\;\Longrightarrow\;
\text{DPM-Solver / DPM-Solver++ 2M}.
$$

接着推导 `noise`、`x_start`、`score` 与 `pf_velocity` 的精确转换、速度等价训练权重，以及为什么 `pf_velocity + DPM-Solver++ 2M` 会病态。代码单元是自包含教学实现，仅依赖 PyTorch；工程实现对应 `schedules/probability_paths.py` 与 `solvers/vp_dpm_solver.py`。


## 1. VP 前向 SDE 的边缘分布从哪里来

取连续时间 variance-preserving SDE：

$$
d x_t=-\frac12\beta(t)x_tdt+\sqrt{\beta(t)}dW_t,
\qquad
\beta(t)=\beta_{\min}+(\beta_{\max}-\beta_{\min})t.
$$

这是线性 SDE。设

$$
\alpha_t=
\exp\left[-\frac12\int_0^t\beta(r)dr\right],
$$

并令 $y_t=x_t/\alpha_t$。因为

$$
d\alpha_t=-\frac12\beta(t)\alpha_tdt,
$$

应用 Itô 乘积法则可得漂移恰好抵消：

$$
dy_t=\frac{\sqrt{\beta(t)}}{\alpha_t}dW_t.
$$

故条件于 $x_0$，$y_t$ 是高斯随机变量，进而

$$
x_t=\alpha_t x_0+\sigma_t\epsilon,
\qquad \epsilon\sim\mathcal N(0,I),
$$

其中

$$
\sigma_t^2
=\alpha_t^2\int_0^t\frac{\beta(r)}{\alpha_r^2}dr
=1-\alpha_t^2.
$$

最后一个等号来自对 $\alpha_t^2$ 求导：

$$
\frac{d\alpha_t^2}{dt}=-\beta(t)\alpha_t^2.
$$

对本工程的线性 beta schedule，积分可显式写成

$$
\log\alpha_t
=-\frac12\beta_{\min}t
-\frac14(\beta_{\max}-\beta_{\min})t^2,
\qquad
\sigma_t=\sqrt{1-\alpha_t^2}.
$$


In [ ]:
import torch


class LinearVPSDE:
    """Continuous VP schedule used by every standalone check in this notebook."""

    def __init__(self, beta_min=0.1, beta_max=20.0):
        self.beta_min = beta_min
        self.beta_max = beta_max

    def beta(self, t):
        return self.beta_min + (self.beta_max - self.beta_min) * t

    def log_alpha(self, t):
        return -0.5 * self.beta_min * t - 0.25 * (self.beta_max - self.beta_min) * t.square()

    def alpha(self, t):
        return self.log_alpha(t).exp()

    def sigma(self, t):
        # clamp_min only protects floating-point roundoff at exactly t=0.
        return torch.sqrt((-torch.expm1(2.0 * self.log_alpha(t))).clamp_min(1e-12))

    def lam(self, t):
        """Half log-SNR: lambda=log(alpha)-log(sigma)."""
        return self.log_alpha(t) - torch.log(self.sigma(t))


schedule = LinearVPSDE()
t = torch.linspace(0.01, 0.99, 20)
# The VP identity alpha^2 + sigma^2 = 1 verifies the closed-form schedule.
torch.testing.assert_close(schedule.alpha(t).square() + schedule.sigma(t).square(),
                           torch.ones_like(t), atol=2e-6, rtol=0)
print("VP schedule identity passed")


## 2. 为什么这个确定性 ODE 与 SDE 具有相同边缘分布

一般 Itô SDE

$$
dx=f(x,t)dt+g(t)dW_t
$$

的密度 $p_t(x)$ 满足 Fokker--Planck 方程

$$
\frac{\partial p_t}{\partial t}
=-\nabla\cdot(f p_t)+\frac12 g(t)^2\Delta p_t.
$$

因为 $\Delta p=\nabla\cdot(p\nabla\log p)$，它可改写成守恒形式

$$
\frac{\partial p_t}{\partial t}
=-\nabla\cdot\left[
\left(f-\frac12g(t)^2\nabla\log p_t\right)p_t
\right].
$$

而确定性 ODE $dx/dt=b(x,t)$ 的密度满足连续性方程

$$
\frac{\partial p_t}{\partial t}=-\nabla\cdot(b p_t).
$$

两式相同，因此取

$$
\boxed{
\frac{dx_t}{dt}=f(x_t,t)-\frac12g(t)^2s_\theta(x_t,t)},
\qquad s_\theta\approx\nabla_x\log p_t(x),
$$

就得到具有相同边缘分布的 probability-flow ODE。VP 情形 $f=-\tfrac12\beta x$、$g^2=\beta$，所以

$$
\boxed{
\frac{dx_t}{dt}
=-\frac12\beta(t)x_t-\frac12\beta(t)s_\theta(x_t,t).}
$$

训练时间从 $0$（数据）走到 $1$（噪声）；生成时沿这个 ODE 从 $t\approx1$ 反向积分到 $t\approx0$。ODE 是确定性的，故相同初始噪声、条件、时间网格下，不同求解器的输出可以一一对应地比较。


## 3. 四种网络输出的严格互换

对条件高斯核

$$
x_t=\alpha_t x_0+\sigma_t\epsilon,
$$

若 $\hat x_0(x_t,t)$ 近似后验均值，Tweedie 关系为

$$
s_\theta(x_t,t)=\frac{\alpha_t\hat x_0-x_t}{\sigma_t^2}.
$$

另一方面，定义噪声预测

$$
\epsilon_\theta=\frac{x_t-\alpha_t\hat x_0}{\sigma_t},
$$

则

$$
s_\theta=-\frac{\epsilon_\theta}{\sigma_t},
\qquad
\hat x_0=\frac{x_t-\sigma_t\epsilon_\theta}{\alpha_t}.
$$

将 score 代回 probability-flow ODE，可得到 noise 表示的速度：

$$
u_\theta
=-\frac12\beta x_t
+\frac12\beta\frac{\epsilon_\theta}{\sigma_t}.
$$

本项目的 `pf_velocity` 正是 $u_\theta$，不是 DDPM 文献常说的 latent-$v$。反解噪声为

$$
\epsilon_\theta
=\sigma_t\left(x_t+\frac{2u_\theta}{\beta(t)}\right).
$$

再消去 $\epsilon_\theta$，直接由 PF velocity 得到 data prediction：

$$
\boxed{
\hat x_0
=\alpha_t x_t-\frac{2\sigma_t^2}{\alpha_t\beta(t)}u_\theta.}
$$

在精确模型下四种表示等价；在有模型误差时，最后一个式子的 $1/\alpha_t$ 是重要的病态来源。


In [ ]:
def epsilon_from_prediction(x, t, raw_prediction, prediction_type, schedule):
    """Convert one of four raw outputs into epsilon; scalar t keeps notation clear."""
    alpha, sigma = schedule.alpha(t), schedule.sigma(t)
    if prediction_type == "noise":
        return raw_prediction
    if prediction_type == "x_start":
        return (x - alpha * raw_prediction) / sigma
    if prediction_type == "score":
        return -sigma * raw_prediction
    if prediction_type == "pf_velocity":
        return sigma * (x + 2.0 * raw_prediction / schedule.beta(t))
    raise ValueError("unknown prediction type")


def pf_velocity_from_noise(x, t, epsilon, schedule):
    """Probability-flow drift, not DDPM latent-v."""
    return -0.5 * schedule.beta(t) * x + 0.5 * schedule.beta(t) * epsilon / schedule.sigma(t)


torch.manual_seed(0)
t = torch.tensor(0.4)
x, epsilon = torch.randn(5, 2), torch.randn(5, 2)
velocity = pf_velocity_from_noise(x, t, epsilon, schedule)
x_start = (x - schedule.sigma(t) * epsilon) / schedule.alpha(t)
score = -epsilon / schedule.sigma(t)

for kind, raw in {"noise": epsilon, "x_start": x_start,
                  "score": score, "pf_velocity": velocity}.items():
    torch.testing.assert_close(epsilon_from_prediction(x, t, raw, kind, schedule), epsilon)
print("noise, x_start, score and PF-velocity conversions agree")


## 4. 为什么训练时需要速度等价加权

“每种输出都直接做 MSE”并不等价，因为同样大小的 raw-output 误差会造成不同大小的 ODE 速度误差。固定 $x_t,t$，对各个表示的误差做一阶传播：

噪声预测中

$$
\delta u=\frac{\beta}{2\sigma}\delta\epsilon;
$$

data prediction 中

$$
\epsilon=\frac{x-\alpha\hat x_0}{\sigma}
\quad\Longrightarrow\quad
\delta u=-\frac{\beta\alpha}{2\sigma^2}\delta\hat x_0;
$$

score prediction 中

$$
\delta u=-\frac\beta2\delta s;
$$

PF velocity 则直接有 $\delta u=\delta u_{\mathrm{raw}}$。因此若目标是让四种模型的局部 ODE 误差尽量可比，应最小化加权 MSE，权重取雅可比平方：

$$
w_{\epsilon}=\left(\frac{\beta}{2\sigma}\right)^2,
\qquad
w_{x_0}=\left(\frac{\beta\alpha}{2\sigma^2}\right)^2,
$$

$$
w_s=\left(\frac\beta2\right)^2,
\qquad
w_{u}=1.
$$

这些权重在端点附近可能很大；工程上须限制训练时间到 $[\varepsilon,1-\varepsilon]$、做权重 clip，并把 batch 平均权重归一到 1。加权只使“速度误差目标”更公平，不能改变某一参数化本身的病态转换。


## 5. 半线性结构到底从哪里来

“半线性”不是额外假设，而是把 probability-flow ODE 中**已知且只含当前状态 $x$ 的线性漂移**，与**需要神经网络估计的 forcing** 分开。网络输出仍依赖 $x$，所以整体 ODE 并不是线性 ODE；但只要在一步积分中把网络输出视为给定函数，线性部分可以精确处理。

### 5.1 从 score 形式改写为 noise forcing

VP probability-flow ODE 是

$$
\frac{dx}{dt}=-\frac12\beta(t)x-\frac12\beta(t)s_\theta(x,t).
$$

使用

$$
s_\theta=-\frac{\epsilon_\theta}{\sigma_t},
$$

可得

$$
\boxed{
\frac{dx}{dt}
=\underbrace{-\frac12\beta(t)x}_{\text{已知线性项}}
+\underbrace{\frac{\beta(t)}{2\sigma_t}\epsilon_\theta(x,t)}_{\text{神经网络 forcing}}.}
$$

这已是标准形式

$$
\frac{dx}{dt}=a(t)x+g_\theta(x,t),
\qquad a(t)=-\frac12\beta(t).
$$

如果暂时把 $g_\theta$ 视为外部输入，齐次方程是

$$
\frac{dx}{dt}=-\frac12\beta(t)x.
$$

分离变量：

$$
\frac{dx}{x}=-\frac12\beta(t)dt,
$$

从 $s$ 积分到 $t$：

$$
x_t=x_s\exp\left[-\frac12\int_s^t\beta(r)dr\right]
=\frac{\alpha_t}{\alpha_s}x_s.
$$

这就是 DPM-Solver 中的线性精确传播比例 $\alpha_t/\alpha_s$。注意它没有被丢弃：每一个积分步都必须乘上该比例。

### 5.2 积分因子如何消去 noise 形式的线性项

定义

$$
y_t=\frac{x_t}{\alpha_t}.
$$

因为

$$
\alpha_t'=-\frac12\beta(t)\alpha_t,
$$

商法则给出

$$
\frac{dy_t}{dt}
=\frac{1}{\alpha_t}\frac{dx_t}{dt}
-\frac{x_t}{\alpha_t^2}\frac{d\alpha_t}{dt}.
$$

代入半线性 ODE：

$$
\frac{dy_t}{dt}
=\frac{1}{\alpha_t}
\left(-\frac12\beta x_t+\frac{\beta}{2\sigma_t}\epsilon_\theta\right)
+\frac{\beta x_t}{2\alpha_t}
=\frac{\beta(t)}{2\alpha_t\sigma_t}\epsilon_\theta(x_t,t).
$$

两项 $-\beta x/(2\alpha)$ 与 $+\beta x/(2\alpha)$ 恰好抵消。这里“消掉”的是变换后方程里的线性导数项；其状态传播作用已被编码进 $x_t=\alpha_t y_t$，并没有从真实轨迹中删除。

### 5.3 改写为 data forcing 后，线性项为什么变成另一种形式

DPM-Solver++ 希望直接使用 clean-data prediction $D_\theta=\hat x_0$。由

$$
\epsilon_\theta=\frac{x_t-\alpha_tD_\theta}{\sigma_t},
$$

代入 noise 形式：

$$
\frac{dx}{dt}
=-\frac12\beta x
+\frac{\beta}{2\sigma_t^2}(x-\alpha_tD_\theta).
$$

因为 $1/\sigma_t^2-1=\alpha_t^2/\sigma_t^2$，整理为

$$
\boxed{
\frac{dx}{dt}
=\underbrace{\frac{\beta(t)\alpha_t^2}{2\sigma_t^2}x}_{\text{已知线性项}}
-\underbrace{\frac{\beta(t)\alpha_t}{2\sigma_t^2}D_\theta(x,t)}_{\text{神经网络 forcing}}.}
$$

这仍然是半线性结构，只是线性系数与 noise 表示不同。验证它的齐次解：

$$
\frac{d\log\sigma_t}{dt}
=\frac{\sigma_t'}{\sigma_t}
=\frac{\beta(t)\alpha_t^2}{2\sigma_t^2}.
$$

因此齐次方程的精确传播为

$$
x_t=\frac{\sigma_t}{\sigma_s}x_s.
$$

这正是 DPM-Solver++ 的线性比例。

### 5.4 积分因子如何消去 data 形式的线性项

令

$$
z_t=\frac{x_t}{\sigma_t}.
$$

由上面的 data 半线性式和

$$
\sigma_t'=\frac{\beta(t)\alpha_t^2}{2\sigma_t},
$$

可得

$$
\begin{aligned}
\frac{dz_t}{dt}
&=\frac1{\sigma_t}\frac{dx_t}{dt}
-\frac{x_t}{\sigma_t^2}\sigma_t'\\
&=\frac{\beta\alpha_t^2}{2\sigma_t^3}x_t
-\frac{\beta\alpha_t}{2\sigma_t^3}D_\theta
-\frac{\beta\alpha_t^2}{2\sigma_t^3}x_t\\
&=-\frac{\beta(t)\alpha_t}{2\sigma_t^3}D_\theta(x_t,t).
\end{aligned}
$$

同样，$x$ 的线性项在 $z=x/\sigma$ 的导数中抵消，而它的精确传播保留在 $x_t=\sigma_tz_t$ 中。

所以 DPM-Solver 与 DPM-Solver++ 的本质差异不是“一个有解析解、另一个没有”，而是对同一 probability-flow ODE 选择了不同的半线性分解：

| 表示 | 归一化状态 | 精确线性传播 | 数值近似的 forcing |
|---|---|---|---|
| DPM-Solver | $x/\alpha$ | $\alpha_t/\alpha_s$ | $\epsilon_\theta$ |
| DPM-Solver++ | $x/\sigma$ | $\sigma_t/\sigma_s$ | $D_\theta=\hat x_0$ |

后续 $lambda$ 变换只是让这两个 forcing 的积分核变成指数函数，从而方便构造稳定的一阶/二阶近似。


## 6. 进入 log-SNR 坐标：DPM-Solver 的半线性结构

定义

$$
\lambda(t)=\log\alpha_t-\log\sigma_t.
$$

先求导。由 $\alpha'=-\tfrac12\beta\alpha$ 和

$$
2\sigma\sigma'=-2\alpha\alpha'=\beta\alpha^2,
$$

得到

$$
\sigma'=\frac{\beta\alpha^2}{2\sigma},
\qquad
\frac{d\lambda}{dt}
=-\frac\beta2-\frac{\beta\alpha^2}{2\sigma^2}
=-\frac{\beta}{2\sigma^2}.
$$

采样时 $t$ 从大到小，因此 $\lambda$ 从小到大。这个坐标不是魔法地“消除网络误差”；它把可解析的线性部分分离出来，并在原始时间端点附近自动形成非均匀网格。

### 5.1 Noise 形式：DPM-Solver

令 $y=x/\alpha$。使用

$$
\frac{dx}{dt}=-\frac12\beta x+\frac12\beta\frac{\epsilon_\theta}{\sigma}
$$

和 $\alpha'=-\tfrac12\beta\alpha$，有

$$
\frac{dy}{dt}=\frac{\beta}{2\alpha\sigma}\epsilon_\theta.
$$

再除以 $d\lambda/dt$：

$$
\boxed{
\frac{d(x/\alpha)}{d\lambda}=-e^{-\lambda}\epsilon_\theta(x_\lambda,\lambda).}
$$

从采样节点 $s$ 积到下一节点 $t$，其中 $\lambda_t>\lambda_s$：

$$
x_t=\frac{\alpha_t}{\alpha_s}x_s
-\alpha_t\int_{\lambda_s}^{\lambda_t}e^{-\lambda}
\epsilon_\theta(x_\lambda,\lambda)d\lambda.
$$

线性比例 $\alpha_t/\alpha_s$ 完全解析；只需近似积分中的网络输出。

### 5.2 Data 形式：DPM-Solver++

令 $z=x/\sigma$，以及 $D_\theta=\hat x_0$。将

$$
\epsilon_\theta=\frac{x-\alpha D_\theta}{\sigma}
$$

代入 ODE 后可得

$$
\frac{dz}{dt}=-\frac{\beta\alpha}{2\sigma^3}D_\theta.
$$

除以 $d\lambda/dt$：

$$
\boxed{
\frac{d(x/\sigma)}{d\lambda}=e^{\lambda}D_\theta(x_\lambda,\lambda).}
$$

因此

$$
x_t=\frac{\sigma_t}{\sigma_s}x_s
+\sigma_t\int_{\lambda_s}^{\lambda_t}e^\lambda
D_\theta(x_\lambda,\lambda)d\lambda.
$$

这就是 DPM-Solver++ 以 data prediction 为非线性 forcing 的来源。它不是在原始 t 空间普通地对速度做 Heun。


## 7. 一阶更新：冻结非线性项，但线性项精确积分

令

$$
h=\lambda_t-\lambda_s>0.
$$

### 6.1 DPM-Solver（noise）

在积分中取 $\epsilon_\theta(\lambda)\approx\epsilon_s$：

$$
\alpha_t\int_{\lambda_s}^{\lambda_t}e^{-\lambda}\epsilon_s d\lambda
=\sigma_t\int_0^h e^rdr\,\epsilon_s
=\sigma_t(e^h-1)\epsilon_s.
$$

故

$$
\boxed{
x_t=\frac{\alpha_t}{\alpha_s}x_s
-\sigma_t\operatorname{expm1}(h)\epsilon_s.}
$$

### 6.2 DPM-Solver++（data）

在积分中取 $D_\theta(\lambda)\approx D_s$：

$$
\sigma_t\int_{\lambda_s}^{\lambda_t}e^\lambda D_s d\lambda
=\alpha_t\int_0^h e^{-r}dr\,D_s
=\alpha_t(1-e^{-h})D_s.
$$

故

$$
\boxed{
x_t=\frac{\sigma_t}{\sigma_s}x_s
-\alpha_t\operatorname{expm1}(-h)D_s.}
$$

`expm1` 计算 $e^h-1$ 或 $e^{-h}-1$，比直接写 `exp(h)-1` 更适合小步长，避免相近数相减造成的有效位损失。


## 8. 二阶多步更新：历史模型输出怎样产生系数

首步没有历史输出，只能用上一节的一阶更新。第二步开始，保存前一个模型值 $m_{s^-}$，并在 $\lambda$ 上做线性外推：

$$
m(\lambda)\approx m_s
+\frac{\lambda-\lambda_s}{h_0}(m_s-m_{s^-}),
\qquad h_0=\lambda_s-\lambda_{s^-}.
$$

令

$$
d_1=\frac{h}{h_0}(m_s-m_{s^-}).
$$

工程中的 2M 公式采用二阶的局部近似

$$
\int_0^h e^r m(\lambda_s+r)dr
\approx \operatorname{expm1}(h)\left(m_s+\frac12d_1\right),
$$

$$
\int_0^h e^{-r}m(\lambda_s+r)dr
\approx -\operatorname{expm1}(-h)\left(m_s+\frac12d_1\right).
$$

所以实际实现是

$$
\boxed{
x_t=\frac{\alpha_t}{\alpha_s}x_s
-\sigma_t\operatorname{expm1}(h)
\left(\epsilon_s+\frac12d_1\right)}
$$

以及

$$
\boxed{
x_t=\frac{\sigma_t}{\sigma_s}x_s
-\alpha_t\operatorname{expm1}(-h)
\left(D_s+\frac12d_1\right).}
$$

这里的 $\tfrac12d_1$ 是二阶截断所保留的项，不是“再调用一次中点 decoder”。因此 2M 在第一步后每区间只需一个新 NFE。 本项目的部署实现还会在最后执行一次 `denoise_to_zero` data projection，因此若有 30 个积分区间，实际 decoder NFE 为 $30+1=31$；这一次不属于二步积分 stage，但必须计入真实推理耗时。若希望对线性 $m(\lambda)$ 的加权积分完全精确，系数应分别是

$$
\frac{e^h(h-1)+1}{h_0}
\quad\text{和}\quad
\frac{1-(h+1)e^{-h}}{h_0};
$$

它们与 2M 的二阶近似在 $h\to0$ 时一致，但有限步长时不相同。本项目保留该区别作为 CondOT `exact_linear` 与 `dpmpp` 消融的原因。


In [ ]:
class DPMTwoStep:
    """Pedagogical VP DPM-Solver / DPM-Solver++ 2M implementation.

    ``prediction`` must provide epsilon(x,t) and x_start(x,t).  The time grid
    is decreasing in physical t, hence increasing in lambda.
    """

    def __init__(self, prediction, schedule, algorithm="dpmsolver"):
        if algorithm not in {"dpmsolver", "dpmsolver++"}:
            raise ValueError("algorithm must be 'dpmsolver' or 'dpmsolver++'")
        self.prediction = prediction
        self.schedule = schedule
        self.algorithm = algorithm
        self.nfe = 0

    def model_value(self, x, t):
        self.nfe += 1
        return (self.prediction.x_start(x, t) if self.algorithm == "dpmsolver++"
                else self.prediction.epsilon(x, t))

    @torch.no_grad()
    def sample(self, x_init, times):
        x = x_init
        old_model_value = None
        old_time = None
        self.nfe = 0
        for index, (s, t) in enumerate(zip(times[:-1], times[1:])):
            model_s = self.model_value(x, s)
            lambda_s, lambda_t = self.schedule.lam(s), self.schedule.lam(t)
            h = lambda_t - lambda_s
            alpha_s, alpha_t = self.schedule.alpha(s), self.schedule.alpha(t)
            sigma_s, sigma_t = self.schedule.sigma(s), self.schedule.sigma(t)

            if index == 0:
                # First-order warm-up: there is no historical model value yet.
                correction = model_s
            else:
                h0 = lambda_s - self.schedule.lam(old_time)
                d1 = (h / h0) * (model_s - old_model_value)
                correction = model_s + 0.5 * d1  # The second-order multistep term.

            if self.algorithm == "dpmsolver":
                x = alpha_t / alpha_s * x - sigma_t * torch.expm1(h) * correction
            else:
                x = sigma_t / sigma_s * x - alpha_t * torch.expm1(-h) * correction

            old_model_value, old_time = model_s, s

        # Match the common denoise-to-zero deployment convention.
        # This is an extra decoder call, so it must be counted in real latency/NFE.
        self.nfe += 1
        return self.prediction.x_start(x, times[-1])


## 9. PF velocity 为什么能直接积分，却可能不适合 DPM-Solver++

PF velocity 是 ODE 的原生右端，因此 Euler、Midpoint、Heun、RK4 和 AB2 都可以直接以它积分，不需要任何表示转换。问题只在于 DPM-Solver++ 的 forcing 是 $D=\hat x_0$。

由第 3 节的关系

$$
\hat x_0=\alpha x-\frac{2\sigma^2}{\alpha\beta}u,
$$

在固定 $x,t$ 处的误差传播为

$$
\boxed{
\delta\hat x_0=-\frac{2\sigma_t^2}{\alpha_t\beta(t)}\delta u.}
$$

采样起点高噪声区 $alpha_t\to0$，这个系数可能很大。于是速度预测中微小且可接受的误差，经转换后会成为巨大的 data-prediction 误差；DPM-Solver++ 的多步外推随之放大该误差。这里的失败不是“PF ODE 不稳定”，而是“从 PF velocity 到 DPM++ 所需 $x_0$ 的表示转换病态”。

工程结论是：

- `pf_velocity` 应配显式速度法或 AB2；
- `noise/score` 自然配 DPM-Solver；
- `x_start` 或稳定的 data prediction 自然配 DPM-Solver++；
- 若同一网络必须服务两类 sampler，使用 `(pf_velocity, x_start)` 双头，而不是强行从速度恢复 $x_0$。


In [ ]:
# Quantify the conditioning factor in the PF-velocity -> x_start conversion.
times = torch.tensor([0.1, 0.5, 0.9, 0.99])
amplification = 2.0 * schedule.sigma(times).square() / (schedule.alpha(times) * schedule.beta(times))
for time, factor in zip(times.tolist(), amplification.tolist()):
    print(f"t={time:>4.2f}: |delta x0| / |delta u| = {factor:8.3f}")

# The factor grows towards the high-noise endpoint, demonstrating the issue.
assert amplification[-1] > amplification[0]
print("conditioning-growth check passed")


## 10. 实验读数应如何解释

固定同一个 checkpoint、同一初始噪声和条件时，`MSE vs RK4(512)` 测量的是数值求解器是否逼近该网络自己的连续 ODE；它适合在同一 prediction 行内比较。不同 prediction checkpoint 有不同的学习误差和不同的连续 ODE，不能跨行把 reference MSE 当作模型质量。

跨参数化、跨路径时应看生成分布质量：本 toy 实验报告每个条件的均值误差、协方差误差、Gaussian Wasserstein-2 distance 和样本图。迁移到真实规划模型时，应以相同初始噪声和 NFE 为前提，再比较碰撞率、进度、舒适性、多样性等下游指标。
